# Завантаження aggTrades: план, прогін, перевірка

**Навіщо цей ноутбук.** У `results/iter7_*.csv` стратегії `cvd_momentum` і `ob_imbalance`
мали `status="ok"` зі Sharpe рівно 0.0 — насправді вони не мали даних: кеш aggTrades
покривав ~2.7 доби з 3 років. Після аудиту такі клітинки позначаються `error`
(«потік 'trades' перекриває лише 0.3% періоду»), а не вдають виміряний нуль.

Цей ноутбук **не качає 3 роки**. Він:

1. рахує, скільки це реально коштує в ГБ і RAM на кожен символ (на ваших даних);
2. показує, які саме архіви будуть завантажені (без мережі);
3. друкує готові команди для терміналу;
4. перевіряє результат — покриття `aggTrade id` по кожному символу;
5. запускає валідацію flow-стратегій на короткому вікні.

**TL;DR про докачку:**

| шлях | поведінка при обриві |
|---|---|
| `--vision` (Binance Vision, історія) | **докачує.** Дані флашаться на диск кожні 5 архівів, а повторний запуск пропускає вже повні дні |
| `--trades` (REST, останні ~2 доби) | **докачує.** Чекпоінт кожні 100 батчів + збереження при обриві |
| згадати, що вже є | `missing_vision_periods()` — див. клітинку нижче |

Запускати заново з нуля **не потрібно**: скрипт сам визначає, чого бракує.

In [1]:
# ── Bootstrap: шляхи, імпорти, налаштування ──────────────────────────────────
import logging
import sys
from datetime import date, timedelta
import warnings
from pathlib import Path

import pandas as pd

# LightGBM тягне `tqdm.auto`, який у ноутбуці без `ipywidgets` пише
# "TqdmWarning: IProgress not found". Це косметика, не помилка виконання;
# глушимо лише це конкретне повідомлення (лікується `uv pip install ipywidgets`).
warnings.filterwarnings("ignore", message="IProgress not found.*")

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(name)s: %(message)s", datefmt="%H:%M:%S")

from scalper_hft.data.binance_vision import complete_trade_days, missing_vision_periods  # noqa: E402
from scalper_hft.data.store import get_store  # noqa: E402
from scalper_hft.validation.walk_forward import MIN_STREAM_COVERAGE  # noqa: E402

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Виміряно на цьому кеші:
#   у RAM pandas-фрейм aggTrades ≈ 43 байти/рядок (trade_id+price+amount+ts+side)
#   на диску (parquet) ≈ 6.5 байти/рядок — стиснення ~6.6×
RAM_BYTES_PER_ROW = 43
DISK_BYTES_PER_ROW = 6.5
store = get_store()
print(f"project_root = {project_root}")
print(f"поріг покриття потоку у walk-forward: {MIN_STREAM_COVERAGE:.0%}")

project_root = /home/volodymyr/PycharmProjects/scalper-hft
поріг покриття потоку у walk-forward: 50%


## 1. Конфігурація

`DAYS` — не «скільки історії взагалі», а **вікно тесту**. `require_stream_coverage`
вимагає, щоб trades покривали ≥50% цього вікна, тож достатньо завантажити рівно його.

Рекомендація: **180 днів**. При `--train 500 --test 200` це ~19 OOS-вікон на 1h.
Для 90 днів вікон буде ~8 — замало для вердикту (ставте тоді `--train 300 --test 100`).

In [2]:
DAYS = 180                      # вікно тесту, днів
TRAIN_BARS = 500                # 1h-барів у train-вікні walk-forward
TEST_BARS = 200                 # 1h-барів у OOS-вікні

# Символи. BTC/ETH найліквідніші, але й найдорожчі (див. оцінку нижче).
#   важкі:  BTCUSDT,ETHUSDT,SOLUSDT,XRPUSDT
#   легкі:  SOLUSDT,XRPUSDT,BNBUSDT,DOGEUSDT   ← рекомендовано
SYMS = ["SOLUSDT", "XRPUSDT", "BNBUSDT", "DOGEUSDT"]

TODAY = date.today()
START = TODAY - timedelta(days=DAYS)
DAILY_FROM = TODAY.replace(day=1)   # місячні дампи мають лаг ~місяць → хвіст добираємо денними
SYMS_STR = ",".join(SYMS)

print(f"сьогодні: {TODAY} | вікно тесту з {START} ({DAYS} днів)")
print(f"символи: {SYMS_STR}")

сьогодні: 2026-09-12 | вікно тесту з 2026-03-16 (180 днів)
символи: SOLUSDT,XRPUSDT,BNBUSDT,DOGEUSDT


## 2. Скільки це коштує (по ваших даних)

Оцінка береться з **вашого кешу**: справжня кількість угод виводиться з діапазону
`trade_id` (він монотонний з кроком 1), а не з кількості рядків — рядків якраз
бракує через старі дефекти дедупу.

> ⚠️ Обмеження сховища: кеш — це **один parquet на символ**, і злиття нового архіву з
> історією вимагає тримати всю історію в RAM. BTCUSDT 180 днів ≈ 8.5 ГБ, пік при
> `concat` ≈ 17 ГБ. Тому важкі символи беріть на коротше вікно, а легкі — на довше.

In [3]:
rows = []
for sym in SYMS:
    d = store.load_trades(sym)
    if d is None or d.empty:
        rows.append({"symbol": sym, "рядків у кеші": 0, "покриття id": None,
                     "угод/добу": None, "МБ на диску": None, "RAM, ГБ": None})
        continue
    ids = d["trade_id"]
    real = ids[ids > 0]
    span_days = max((d.index.max() - d.index.min()).total_seconds() / 86400, 1e-9)
    true_ids = int(real.max() - real.min() + 1) if len(real) > 1 else len(real)
    per_day = true_ids / span_days
    rows.append({
        "symbol": sym,
        "рядків у кеші": len(d),
        "покриття id": (len(real) / true_ids) if true_ids else None,
        "угод/добу": int(per_day),
        "МБ на диску": per_day * DISK_BYTES_PER_ROW * DAYS / 1e6,
        "RAM, ГБ": per_day * RAM_BYTES_PER_ROW * DAYS / 1e9,
    })

est = pd.DataFrame(rows)
fmt = {"рядків у кеші": "{:,.0f}".format,
       "покриття id": lambda v: "—" if v is None else f"{v:.1%}",
       "угод/добу": lambda v: "—" if v is None else f"{v:,.0f}",
       "МБ на диску": lambda v: "—" if v is None else f"{v:,.0f}",
       "RAM, ГБ": lambda v: "—" if v is None else f"{v:.1f}"}
print(est.to_string(index=False, formatters=fmt))
if not est.empty and est["МБ на диску"].notna().any():
    print(f"\nРазом на диску: ~{est['МБ на диску'].dropna().sum() / 1000:.1f} ГБ")
    ram = est["RAM, ГБ"].dropna()
    if not ram.empty:
        print(f"RAM на символ (пік ×2 при злитті): {ram.min():.1f}–{ram.max():.1f} ГБ → до {2 * ram.max():.1f} ГБ пік")
        if 2 * ram.max() > 16:
            print("⚠️  ризик OOM: зменште DAYS або приберіть важкі символи")

  symbol рядків у кеші покриття id угод/добу МБ на диску RAM, ГБ
 SOLUSDT    27,884,579       52.5%   274,187         321     2.1
 XRPUSDT       606,601       93.9%   241,522         283     1.9
 BNBUSDT       525,009       69.2%   274,749         321     2.1
DOGEUSDT       478,148       95.2%   183,959         215     1.4

Разом на диску: ~1.1 ГБ
RAM на символ (пік ×2 при злитті): 1.4–2.1 ГБ → до 4.3 ГБ пік


## 3. Що саме буде завантажено (без мережі)

`missing_vision_periods()` — та сама функція, яку використовує завантажувач. Вона показує
**точно**, які архіви підуть у запит. Порожньо → качати нічого.

День вважається вже завантаженим, якщо він має і часовий проміжок ≥20 год, **і** майже
всі `aggTrade id` без розривів. Друга умова критична: ваш кеш має повні проміжки, але
39% id — саме тому зіпсовані дні тепер перекачуються, а не пропускаються.

> У логах побачите `день … неповний за aggTrade id — перекачаю` — це очікувано
> і саме те, що потрібно: завантажувач помітив діри й не вважає такий день готовим.

## 4. Команди для терміналу

**Запускайте в терміналі (`tmux`/фоном), не в ноутбуці** — це години на каналі, і ноутбук
був би заблокований. Клітинка нижче лише друкує готові команди.

Порядок обов'язковий: місячні → денні → REST. Денний крок добирає хвіст поточного місяця
і заодно перекачує зіпсовані дні.

`--vision-checkpoint-every 1` для місячних означає «зберегти кеш після кожного архіву»:
обрив коштує щонайбільше один місяць роботи, і повторний запуск продовжить із місця обриву.
Для денних архівів залишайте типовий (5) — інакше переписування кеша після кожного дня
дасть зайвий I/O на 180 файлів.

In [4]:
CMD = '''cd {root}
SYMS={syms}

# 1) історія — місячні дампи (~{months} архівів на символ). Чекпоінт після кожного архіву:
#    обрив коштує щонайбільше один місяць, повторний запуск продовжить із місця обриву.
uv run python -m scalper_hft.cli download --symbol $SYMS --vision --vision-freq monthly --vision-start {start} --vision-checkpoint-every 1 --days 1

# 2) хвіст поточного місяця — денні дампи (тут типовий чекпоінт кожні 5 архівів)
uv run python -m scalper_hft.cli download --symbol $SYMS --vision --vision-freq daily --vision-start {daily_from} --days 1

# 3) останні ~2 доби — тільки REST (дампів для них не існує)
uv run python -m scalper_hft.cli download --symbol $SYMS --trades --trades-days 2 --days 1
'''
print(CMD.format(root=project_root, syms=SYMS_STR, months=DAYS // 30 + 1, start=START, daily_from=DAILY_FROM))
print("-" * 110)
print("Один символ (перевірити на малому):")
one = next(
    line
    for line in CMD.splitlines()
    if line.startswith("uv run python -m scalper_hft.cli download --symbol $SYMS --vision")
).replace("$SYMS", SYMS[0]).replace("{start}", str(START))
print(one)


cd /home/volodymyr/PycharmProjects/scalper-hft
SYMS=SOLUSDT,XRPUSDT,BNBUSDT,DOGEUSDT

# 1) історія — місячні дампи (~7 архівів на символ). Чекпоінт після кожного архіву:
#    обрив коштує щонайбільше один місяць, повторний запуск продовжить із місця обриву.
uv run python -m scalper_hft.cli download --symbol $SYMS --vision --vision-freq monthly --vision-start 2026-03-16 --vision-checkpoint-every 1 --days 1

# 2) хвіст поточного місяця — денні дампи (тут типовий чекпоінт кожні 5 архівів)
uv run python -m scalper_hft.cli download --symbol $SYMS --vision --vision-freq daily --vision-start 2026-09-01 --days 1

# 3) останні ~2 доби — тільки REST (дампів для них не існує)
uv run python -m scalper_hft.cli download --symbol $SYMS --trades --trades-days 2 --days 1

--------------------------------------------------------------------------------------------------------------
Один символ (перевірити на малому):
uv run python -m scalper_hft.cli download --symbol SOLUSDT --vision --vision-freq month

In [5]:
# Запуск із ноутбука — розкоментуйте свідомо (заблокує ноутбук на час завантаження).
# Для одного символу це прийнятно як перевірка, що все працює:
#
# !uv run python -m scalper_hft.cli download --symbol SOLUSDT \
#   --vision --vision-freq monthly --vision-start 2026-03-15 --days 1

## 5. Перевірка після завантаження

`data-audit` **aggTrades не перевіряє взагалі** — у `scalper_hft/data/audit.py` немає
жодної згадки про trades. Тому дивимось покриття `aggTrade id` напряму.

Ціль: покриття **≈100%** по кожному символу і хвіст не старіший за ~2 доби.

In [6]:
def coverage_report(syms=None) -> pd.DataFrame:
    now = pd.Timestamp.now("UTC").tz_localize(None)
    out = []
    for sym in (syms or SYMS):
        d = store.load_trades(sym)
        if d is None or d.empty:
            out.append({"symbol": sym, "рядків": 0, "покриття id": None,
                        "перший": None, "останній": None, "лаг, год": None})
            continue
        ids = d["trade_id"]
        real = ids[ids > 0]
        cov = len(real) / (real.max() - real.min() + 1) if len(real) > 1 else float("nan")
        out.append({"symbol": sym, "рядків": len(d), "покриття id": cov,
                    "перший": d.index[0], "останній": d.index[-1],
                    "лаг, год": (now - d.index[-1]).total_seconds() / 3600})
    return pd.DataFrame(out)


cov = coverage_report()
print(cov.to_string(index=False, formatters={
    "рядків": "{:,.0f}".format,
    "покриття id": lambda v: "—" if v is None or v != v else f"{v:.1%}",
    "лаг, год": lambda v: "—" if v is None else f"{v:.1f}",
}))
bad = cov[cov["покриття id"].notna() & (cov["покриття id"] < 0.99)]
print("\n✅ усі символи повні" if bad.empty else f"\n⚠️  неповні: {', '.join(bad['symbol'])} — повторіть крок 2")

  symbol     рядків покриття id                  перший                останній лаг, год
 SOLUSDT 27,884,579       52.5% 2026-03-01 00:00:00.010 2026-09-10 17:22:38.726     38.0
 XRPUSDT    606,601       93.9% 2026-09-08 17:30:51.278 2026-09-11 09:42:50.311     21.6
 BNBUSDT    525,009       69.2% 2026-09-08 15:24:21.891 2026-09-11 09:41:39.338     21.6
DOGEUSDT    478,148       95.2% 2026-09-08 16:14:08.481 2026-09-11 09:47:30.936     21.5

⚠️  неповні: SOLUSDT, XRPUSDT, BNBUSDT, DOGEUSDT — повторіть крок 2


## 6. Валідація flow-стратегій на цьому вікні

Окремий прогін, **не** в матрицю на 3 роки. Що очікувати:

- `single:cvd_momentum`, `single:ob_imbalance` → `ok` (або `degenerate`, якщо угод <30).
  Будь-який із цих статусів — **справжній вимір**, на відміну від колишнього `ok` зі Sharpe 0.0;
- `single:supertrend` — контроль: він від trades не залежить;
- `meta:ens_vote` — майже напевно `degenerate` (0 угод): це властивість стратегії
  (один із дітей не торгує → голосування не спрацьовує), а не дефект даних.

In [7]:
VALIDATE = f'''cd {project_root}
uv run python experiments/iter7_regime_rating.py \\
  --days {DAYS} --train {TRAIN_BARS} --test {TEST_BARS} --workers 4 \\
  --symbols {SYMS_STR} \\
  --variants single:cvd_momentum,single:ob_imbalance,single:supertrend,meta:ens_vote \\
  --out-csv results/iter7_flow_shard.csv \\
  --oos-dir results/iter7_flow_oos
'''
print(VALIDATE)
print("Якщо побачите 'PermissionError: [Errno 13]' — приберіть --workers (працюватиме серіально).")

cd /home/volodymyr/PycharmProjects/scalper-hft
uv run python experiments/iter7_regime_rating.py \
  --days 180 --train 500 --test 200 --workers 4 \
  --symbols SOLUSDT,XRPUSDT,BNBUSDT,DOGEUSDT \
  --variants single:cvd_momentum,single:ob_imbalance,single:supertrend,meta:ens_vote \
  --out-csv results/iter7_flow_shard.csv \
  --oos-dir results/iter7_flow_oos

Якщо побачите 'PermissionError: [Errno 13]' — приберіть --workers (працюватиме серіально).


In [8]:
# Прогін із ноутбука (коротке вікно = кілька хвилин):
# !uv run python experiments/iter7_regime_rating.py \
#   --days 180 --train 500 --test 200 --workers 4 \
#   --symbols SOLUSDT,XRPUSDT,BNBUSDT,DOGEUSDT \
#   --variants single:cvd_momentum,single:ob_imbalance,single:supertrend,meta:ens_vote \
#   --out-csv results/iter7_flow_shard.csv --oos-dir results/iter7_flow_oos

In [9]:
# Читання результату (після прогону)
p = project_root / "results" / "iter7_flow_shard.csv"
if p.exists():
    res = pd.read_csv(p)
    cols = [c for c in ["symbol", "variant", "status", "n_trades_oos", "avg_oos_sharpe",
                        "oos_positive_frac", "pooled_oos_sharpe"] if c in res.columns]
    print(res[cols].to_string(index=False))
    print("\nСтатуси:", res["status"].value_counts().to_dict())
    if "error" in res.columns:
        errs = res[res["status"] != "ok"][["variant", "status", "error"]].drop_duplicates("variant")
        if not errs.empty:
            print("\nПричини не-ok:")
            print(errs.to_string(index=False))
else:
    print(f"ще немає {p.relative_to(project_root)} — спершу виконайте крок 6")

ще немає results/iter7_flow_shard.csv — спершу виконайте крок 6


## 7. Застереження

1. **Не знижуйте `min_stream_coverage`**, щоб клітинка «пройшла». Це рівно той фальшивий
   нуль, який прибрали: поріг існує, щоб стратегія без даних падала, а не звітувала Sharpe 0.0.
2. **Не вказуйте `--oos-dir results/iter7_oos`** — перезапишете основні 3-річні артефакти.
   Для flow-валідації завжди окремі `results/iter7_flow_*`.
3. **Скрипт аналізу на цей прогін не запускається**: `iter7_regime_analysis.py` читає
   `results/iter7_oos` (жорстко в коді) і glob `results/iter7_shard_*.csv`. Для короткого
   вікна switch/DSR-аналіз сенсу не має — достатньо CSV рейтингу.
4. **Місячний архів качається цілим**, навіть якщо потрібна частина місяця. Це нормально.
5. **RAM.** Кеш — один parquet на символ, злиття вимагає всієї історії в пам'яті
   (~43 Б/рядок). BTCUSDT 180 днів ≈ 8.5 ГБ, пік ≈ 17 ГБ. Для важких символів зменшуйте
   `DAYS` (90 → ≈4.3 ГБ) або беріть легші (SOL/XRP/BNB/DOGE). Кардинальне рішення —
   партиційні файли замість одного parquet; це зміна сховища, не завантажувача.
6. **Докачка — на рівні символу і дня.** Усередині символу дані флашаться кожні 5 архівів
   (`checkpoint_every`), а повторний запуск пропускає повні дні. Обрив на 3-му архіві з 6
   коштує щонайбільше 2 архіви роботи.